# 2.4 · Stacks and queues

*Sequences* · 20 exercises

Restrict access to one end or both, and use the restriction to match, undo, schedule and evaluate.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-2.4) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-2.4) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-2.4)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
from collections import deque

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 2.4.1 · Stack  `Core`

**Task.** Build `Stack` on a Python list, exposing only `push`, `pop`, `read` (peek), `is_empty` and `__len__`. `pop` and `read` return `None` when empty.

**Example.** push 1, 2, 3 -> `read()` is `3`, `pop()` is `3`, `read()` is `2`

**Big O.** O(1) for every operation. The end of a list is the cheap end.

**Note.** A stack is an abstract data type: the constraint (last in, first out) matters, not the list underneath.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-1)

In [ ]:
# 2.4.1 Stack
class Stack:
    def __init__(self):
        raise NotImplementedError

    def push(self, value):
        raise NotImplementedError

    def pop(self):
        raise NotImplementedError

    def read(self):
        raise NotImplementedError

    def is_empty(self):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

In [ ]:
def test_ex_2_4_1():
    s = Stack()
    assert s.is_empty()
    for v in (1, 2, 3):
        s.push(v)
    assert s.read() == 3
    assert s.pop() == 3
    assert s.read() == 2
    assert len(s) == 2
    assert not s.is_empty()
    assert Stack().pop() is None


test_ex_2_4_1()
print('2.4.1 passed')

### 2.4.2 · Queue  `Core`

**Task.** Build `Queue` with `enqueue`, `dequeue`, `read` (front), `is_empty`, `__len__`. Use `collections.deque`, then explain why `list.pop(0)` would be a poor choice.

**Example.** enqueue 1, 2, 3 -> `dequeue()` is `1`, `read()` is `2`

**Big O.** O(1) each with a deque. `list.pop(0)` is O(N) because every element shifts left (see 2.1.11).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-2)

In [ ]:
# 2.4.2 Queue
class Queue:
    def __init__(self):
        raise NotImplementedError

    def enqueue(self, value):
        raise NotImplementedError

    def dequeue(self):
        raise NotImplementedError

    def read(self):
        raise NotImplementedError

    def is_empty(self):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

In [ ]:
def test_ex_2_4_2():
    q = Queue()
    assert q.is_empty()
    for v in (1, 2, 3):
        q.enqueue(v)
    assert q.dequeue() == 1
    assert q.read() == 2
    assert len(q) == 2
    assert not q.is_empty()


test_ex_2_4_2()
print('2.4.2 passed')

### 2.4.3 · Do the tags nest?  `Core`

**Task.** Write `tags_balanced(markup)` using your `Stack`. Tags are `<name>` and `</name>` with letters only and no attributes; text between tags is ignored. Return `True` when every tag closes, in the reverse order it opened.

**Example.** `'<p><b>hi</b></p>'` -> `True`; `'<p><b></p></b>'` -> `False`

**Big O.** O(N) time, O(N) space for the stack.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-3)

In [ ]:
# 2.4.3 Do the tags nest?
def tags_balanced(markup):
    raise NotImplementedError

In [ ]:
def test_ex_2_4_3():
    assert tags_balanced('<p><b>hi</b></p>')
    assert not tags_balanced('<p><b></p></b>')
    assert not tags_balanced('<p>')
    assert not tags_balanced('</p>')
    assert tags_balanced('plain text')
    assert tags_balanced('<a><b></b><c>x</c></a>')
    assert not tags_balanced('<a></a></a>')


test_ex_2_4_3()
print('2.4.3 passed')

### 2.4.4 · Simplify a file path  `Core`

**Task.** Write `simplify_path(path)` for an absolute path: drop empty parts and `.`, let `..` remove the folder before it (if any), and return the canonical path. Use your `Stack` for the folders.

**Example.** `'/a/./b/../c/'` -> `'/a/c'`; `'/../'` -> `'/'`

**Big O.** O(N) time and space.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-4)

In [ ]:
# 2.4.4 Simplify a file path
def simplify_path(path):
    raise NotImplementedError

In [ ]:
def test_ex_2_4_4():
    assert simplify_path('/a/./b/../c/') == '/a/c'
    assert simplify_path('/../') == '/'
    assert simplify_path('/home//user/') == '/home/user'
    assert simplify_path('/a/b/../../..') == '/'
    assert simplify_path('/x/y/./z') == '/x/y/z'


test_ex_2_4_4()
print('2.4.4 passed')

### 2.4.5 · Round-robin turns  `Core`

**Task.** Write `round_robin(names, times, slice_)` using your `Queue`: job `names[i]` needs `times[i]` units of work. The front job runs for up to `slice_` units; if it still has work left it goes to the back. Return the names in the order the jobs finish.

**Example.** `['a', 'b', 'c'], [3, 1, 2], 1` -> `['b', 'c', 'a']`

**Big O.** O(T / s + N) queue operations for total work T, slice s and N jobs; each one is O(1).

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-5)

In [ ]:
# 2.4.5 Round-robin turns
def round_robin(names, times, slice_):
    raise NotImplementedError

### 2.4.6 · Queue built from two stacks  `Practice`

**Task.** Build `StackQueue` with `enqueue` and `dequeue` using only two `Stack` objects.

**Example.** enqueue 1, 2; dequeue -> `1`; enqueue 3; dequeue -> `2`, dequeue -> `3`

**Big O.** O(1) amortized: each element is moved from the inbox to the outbox at most once.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-6)

In [ ]:
# 2.4.6 Queue built from two stacks
class StackQueue:
    def __init__(self):
        raise NotImplementedError

    def enqueue(self, value):
        raise NotImplementedError

    def dequeue(self):
        raise NotImplementedError

In [ ]:
def test_ex_2_4_6():
    sq = StackQueue()
    sq.enqueue(1)
    sq.enqueue(2)
    assert sq.dequeue() == 1
    sq.enqueue(3)
    assert sq.dequeue() == 2
    assert sq.dequeue() == 3
    assert sq.dequeue() is None


test_ex_2_4_6()
print('2.4.6 passed')

### 2.4.7 · Evaluate postfix (RPN)  `Practice`

**Task.** Write `eval_rpn(tokens)` for `+ - * /` (true division) using a stack.

**Example.** `['3', '4', '+', '2', '*']` -> `14`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-7)

In [ ]:
# 2.4.7 Evaluate postfix (RPN)
def eval_rpn(tokens):
    raise NotImplementedError

In [ ]:
def test_ex_2_4_7():
    assert eval_rpn(['3', '4', '+', '2', '*']) == 14
    assert eval_rpn(['10', '4', '-']) == 6
    assert eval_rpn(['9', '2', '/']) == 4.5


test_ex_2_4_7()
print('2.4.7 passed')

### 2.4.8 · Undo and redo  `Practice`

**Task.** Build `Editor` with `type(text)`, `undo()`, `redo()` and a `text` attribute. Use one stack of past states and one of undone states; new typing clears redo.

**Example.** type 'a', type 'b', undo -> `'a'`; redo -> `'ab'`

**Big O.** O(1) per operation (plus the cost of building the new string).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-8)

In [ ]:
# 2.4.8 Undo and redo
class Editor:
    def __init__(self):
        raise NotImplementedError

    def type(self, chars):
        raise NotImplementedError

    def undo(self):
        raise NotImplementedError

    def redo(self):
        raise NotImplementedError

In [ ]:
def test_ex_2_4_8():
    e = Editor()
    e.type('a')
    e.type('b')
    assert e.undo() == 'a'
    assert e.redo() == 'ab'
    e.undo()
    e.type('c')
    assert e.text == 'ac'
    assert e.redo() == 'ac'
    assert e.undo() == 'a'
    assert e.undo() == ''
    assert e.undo() == ''


test_ex_2_4_8()
print('2.4.8 passed')

### 2.4.9 · Fixed-size circular queue  `Stretch`

**Task.** Build `CircularQueue(capacity)` on a fixed list with `head` and `size`, so neither `enqueue` nor `dequeue` shifts anything. `enqueue` returns `False` when full.

**Example.** capacity 3: enqueue 1, 2, 3 -> 4th returns `False`; dequeue -> `1`; enqueue 4 succeeds and wraps to slot 0

**Big O.** O(1) for both, with no shifting, because the index wraps with `% capacity`.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-9) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-9)

In [ ]:
# 2.4.9 Fixed-size circular queue
class CircularQueue:
    def __init__(self, capacity):
        raise NotImplementedError

    def enqueue(self, value):
        raise NotImplementedError

    def dequeue(self):
        raise NotImplementedError

In [ ]:
def test_ex_2_4_9():
    c = CircularQueue(3)
    assert all((c.enqueue(v) for v in (1, 2, 3)))
    assert not c.enqueue(4)
    assert c.dequeue() == 1
    assert c.enqueue(4)
    assert c.data[0] == 4
    assert [c.dequeue() for _ in range(4)] == [2, 3, 4, None]


test_ex_2_4_9()
print('2.4.9 passed')

### 2.4.10 · Min-max stack  `Core`

**Task.** Build `MinMaxStack` with `push(value)`, `pop()`, `peek()`, `min()` and `max()`, every one O(1). Store each value together with the minimum and maximum of the stack at the moment it was pushed. Callers never pop or read an empty stack.

**Example.** push 5, 2, 8 -> `min()` is `2`, `max()` is `8`; `pop()` -> `8`; `max()` is `5`

**Big O.** O(1) per operation, O(N) space: each entry stores the minimum and maximum of itself and everything below it, so popping restores the old ones for free.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-10) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-10)

In [ ]:
# 2.4.10 Min-max stack
class MinMaxStack:
    def __init__(self):
        raise NotImplementedError

    def push(self, value):
        raise NotImplementedError

    def pop(self):
        raise NotImplementedError

    def peek(self):
        raise NotImplementedError

    def min(self):
        raise NotImplementedError

    def max(self):
        raise NotImplementedError

In [ ]:
def test_ex_2_4_5():
    assert round_robin(['a', 'b', 'c'], [3, 1, 2], 1) == ['b', 'c', 'a']
    assert round_robin(['a', 'b', 'c'], [3, 1, 2], 5) == ['a', 'b', 'c']
    assert round_robin(['x', 'y'], [4, 4], 2) == ['x', 'y']
    assert round_robin([], [], 3) == []


test_ex_2_4_5()
print('2.4.5 passed')

In [ ]:
def test_ex_2_4_10():
    s = MinMaxStack()
    for v in (5, 2, 8):
        s.push(v)
    assert (s.min(), s.max(), s.peek()) == (2, 8, 8)
    assert s.pop() == 8
    assert (s.min(), s.max()) == (2, 5)
    assert s.pop() == 2
    assert (s.min(), s.max()) == (5, 5)
    s.push(5)
    s.push(-1)
    assert (s.min(), s.max(), s.peek()) == (-1, 5, -1)


test_ex_2_4_10()
print('2.4.10 passed')

### 2.4.11 · Next greater value  `Core`

**Task.** Write `next_greater(nums)` that returns, for each index, the first value to its right that is larger, or `-1`. Keep a stack of indices still waiting; each new value settles every waiting index with a smaller value.

**Example.** `[2, 7, 3, 5, 4, 6, 8]` -> `[7, 8, 5, 6, 6, 8, -1]`

**Big O.** O(N) time and space: each index is pushed and popped at most once, and the stack always holds indices still waiting for a larger value.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-11) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-11)

In [ ]:
# 2.4.11 Next greater value
def next_greater(nums):
    raise NotImplementedError

In [ ]:
def test_ex_2_4_11():
    assert next_greater([2, 7, 3, 5, 4, 6, 8]) == [7, 8, 5, 6, 6, 8, -1]
    assert next_greater([]) == []
    assert next_greater([5, 4, 3]) == [-1, -1, -1]
    assert next_greater([1, 1, 2]) == [2, 2, -1]


test_ex_2_4_11()
print('2.4.11 passed')

### 2.4.12 · Sunset view  `Core`

**Task.** `heights` lists buildings from west to east, and the sun sets in the east. Write `sunset_view(heights)` that returns, in ascending order, the indices of the buildings strictly taller than every building east of them. Scan west to east with a stack, popping every building that is not taller than the new one.

**Example.** `[3, 5, 4, 4, 3, 1, 3, 2]` -> `[1, 3, 6, 7]`

**Big O.** O(N) time and space: a building that is not taller than a later one can never see past it, so it leaves the stack for good.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-12) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-12)

In [ ]:
# 2.4.12 Sunset view
def sunset_view(heights):
    raise NotImplementedError

In [ ]:
def test_ex_2_4_12():
    assert sunset_view([3, 5, 4, 4, 3, 1, 3, 2]) == [1, 3, 6, 7]
    assert sunset_view([]) == []
    assert sunset_view([1, 2, 3]) == [2]
    assert sunset_view([3, 2, 1]) == [0, 1, 2]
    assert sunset_view([2, 2]) == [1]


test_ex_2_4_12()
print('2.4.12 passed')

### 2.4.13 · Largest rectangle under a bar chart  `Stretch`

**Task.** `heights` are the bars of a chart, each 1 wide. Write `largest_rectangle(heights)` that returns the largest rectangle area that fits under the bars. Keep a stack of `(start, height)` with rising heights; a lower bar pops every taller entry, scoring `height * (i - start)`, and inherits the start of the last one popped. A final zero-height bar empties the stack.

**Example.** `[2, 4, 3, 5, 1, 3]` -> `9` (height 3 across the bars `4, 3, 5`)

**Big O.** O(N) time and space: a bar is popped exactly when the first lower bar to its right appears, and the entry below it says how far left it reaches.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-13) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-13)

In [ ]:
# 2.4.13 Largest rectangle under a bar chart
def largest_rectangle(heights):
    raise NotImplementedError

In [ ]:
def test_ex_2_4_13():
    assert largest_rectangle([2, 4, 3, 5, 1, 3]) == 9
    assert largest_rectangle([2, 1, 2]) == 3
    assert largest_rectangle([]) == 0
    assert largest_rectangle([5]) == 5
    assert largest_rectangle([1, 1, 1, 1]) == 4
    assert largest_rectangle([2, 3, 4, 5]) == 9


test_ex_2_4_13()
print('2.4.13 passed')

### 2.4.14 · Sort a stack with one more stack  `Practice`

**Task.** A stack is a list with its top at the end. Write `sort_stack(stack)` that empties `stack` into a new stack whose values rise from bottom to top (largest on top) and returns it, using only push, pop and a look at the top of the two stacks: pop a value, move every larger value from the new stack back onto the old one, then push it.

**Example.** push 3, 1, 4, 2 -> popping the result gives `4, 3, 2, 1`

**Big O.** O(N^2) time, O(N) space for the second stack: each value may have to move every larger value aside before it settles.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-14) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-14)

In [ ]:
# 2.4.14 Sort a stack with one more stack
def sort_stack(stack):
    raise NotImplementedError

In [ ]:
def test_ex_2_4_14():
    stack = [3, 1, 4, 2]
    out = sort_stack(stack)
    assert stack == []
    assert [out.pop() for _ in range(4)] == [4, 3, 2, 1]
    assert sort_stack([]) == []
    assert sort_stack([2, 2, 1]) == [1, 2, 2]
    assert sort_stack([1, 2, 3]) == [1, 2, 3]


test_ex_2_4_14()
print('2.4.14 passed')

### 2.4.15 · Collisions on a track  `Practice`

**Task.** `carts` lists carts on a track from left to right: the size is the absolute value, and the sign is the direction (positive moves right). When two meet, the smaller one is destroyed; equal sizes destroy each other. Write `after_collisions(carts)` that returns the carts left, in order. A stack of survivors works: only a left-mover arriving at a right-mover on top collides.

**Example.** `[-3, 4, 9, -6, 2, -2, 7]` -> `[-3, 4, 9, 7]`

**Big O.** O(N) time and space: every cart is pushed once and destroyed at most once.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-15) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-15)

In [ ]:
# 2.4.15 Collisions on a track
def after_collisions(carts):
    raise NotImplementedError

In [ ]:
def test_ex_2_4_15():
    assert after_collisions([-3, 4, 9, -6, 2, -2, 7]) == [-3, 4, 9, 7]
    assert after_collisions([4, 9, -6, 2, -2, -10]) == [-10]
    assert after_collisions([5, -5]) == []
    assert after_collisions([-1, 1]) == [-1, 1]
    assert after_collisions([]) == []


test_ex_2_4_15()
print('2.4.15 passed')

### 2.4.16 · Window maximum with a deque  `Practice`

**Task.** Write `window_max(nums, k)` that returns the maximum of every window of `k` consecutive values (`1 <= k <= len(nums)`), in O(N). Keep a `deque` of indices whose values fall from front to back: drop smaller values from the back before adding, and drop the front once it leaves the window.

**Example.** `[4, 2, 12, 3, 8, 1, 7], 3` -> `[12, 12, 12, 8, 8]`

**Big O.** O(N) time, O(K) space: the deque keeps indices of falling values, so its front is the window's maximum, and each index enters and leaves once.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-16) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-16)

In [ ]:
# 2.4.16 Window maximum with a deque
def window_max(nums, k):
    raise NotImplementedError

In [ ]:
def test_ex_2_4_16():
    assert window_max([4, 2, 12, 3, 8, 1, 7], 3) == [12, 12, 12, 8, 8]
    assert window_max([3, 1, 2], 1) == [3, 1, 2]
    assert window_max([3, 1, 2], 3) == [3]
    assert window_max([5, 4, 3, 2, 1], 2) == [5, 4, 3, 2]
    assert window_max([1, 3, 3, 1], 2) == [3, 3, 3]


test_ex_2_4_16()
print('2.4.16 passed')

### 2.4.17 · Longest balanced bracket run  `Practice`

**Task.** `s` holds only `(` and `)`. Write `longest_balanced(s)` that returns the length of the longest substring of correctly matched brackets. Keep a stack of indices starting with `-1`: push each `(`; on `)` pop, then either measure from the new top or, if the stack is empty, push this index as the new boundary.

**Example.** `"(()())())("` -> `8`

**Big O.** O(N) time and space: the stack holds open-bracket indices above the index of the last unmatched close, which marks where a balanced run can start.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-17) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-17)

In [ ]:
# 2.4.17 Longest balanced bracket run
def longest_balanced(s):
    raise NotImplementedError

In [ ]:
def test_ex_2_4_17():
    assert longest_balanced('(()())())(') == 8
    assert longest_balanced('') == 0
    assert longest_balanced(')(') == 0
    assert longest_balanced('()(()') == 2
    assert longest_balanced('(()') == 2
    assert longest_balanced('()()') == 4
    assert longest_balanced('))()(())') == 6


test_ex_2_4_17()
print('2.4.17 passed')

### 2.4.18 · Stack from a queue  `Core`

**Task.** Build `QueueStack` with `push(value)`, `pop()`, `peek()` and `is_empty()` on one `deque` used only as a queue (`append`, `popleft`, `[0]`, `len`). After appending, rotate the older values from the front to the back so the newest is in front. Callers never pop or peek an empty stack.

**Example.** push 1, 2, 3; `pop()` -> `3`; `peek()` -> `2`

**Big O.** O(N) push, O(1) pop and peek: after each push the queue is rotated so the newest value is at the front.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-18) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-18)

In [ ]:
# 2.4.18 Stack from a queue
class QueueStack:
    def __init__(self):
        raise NotImplementedError

    def push(self, value):
        raise NotImplementedError

    def pop(self):
        raise NotImplementedError

    def peek(self):
        raise NotImplementedError

    def is_empty(self):
        raise NotImplementedError

In [ ]:
def test_ex_2_4_18():
    s = QueueStack()
    assert s.is_empty()
    for v in (1, 2, 3):
        s.push(v)
    assert s.pop() == 3
    assert s.peek() == 2
    assert not s.is_empty()
    s.push(4)
    assert [s.pop(), s.pop(), s.pop()] == [4, 2, 1]
    assert s.is_empty()


test_ex_2_4_18()
print('2.4.18 passed')

### 2.4.19 · Reverse the first k of a queue  `Core`

**Task.** Write `reverse_first_k(queue, k)` that reverses the first `k` values of a `deque` in place (`0 <= k <= len(queue)`) and returns it, using only queue operations (`append`, `popleft`) and a list as a stack: move `k` values onto the stack, append them back, then rotate the remaining `len - k` values from the front to the back.

**Example.** `[1, 2, 3, 4, 5], 3` -> `[3, 2, 1, 4, 5]`

**Big O.** O(N) time, O(K) space for the stack: k values go through the stack, then the other N - k rotate to the back once.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-19) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-19)

In [ ]:
# 2.4.19 Reverse the first k of a queue
def reverse_first_k(queue, k):
    raise NotImplementedError

In [ ]:
def test_ex_2_4_19():
    q = deque([1, 2, 3, 4, 5])
    assert reverse_first_k(q, 3) is q
    assert list(q) == [3, 2, 1, 4, 5]
    assert list(reverse_first_k(deque([1, 2, 3]), 0)) == [1, 2, 3]
    assert list(reverse_first_k(deque([1, 2, 3]), 3)) == [3, 2, 1]
    assert list(reverse_first_k(deque(), 0)) == []


test_ex_2_4_19()
print('2.4.19 passed')

### 2.4.20 · Binary numbers 1 to n with a queue  `Core`

**Task.** Write `binary_numbers(n)` that returns the binary forms of `1` to `n` as strings, in order, without converting numbers: start a `deque` with `"1"`, and each time you take a string from the front, record it and add it with `"0"` and `"1"` appended to the back.

**Example.** `5` -> `["1", "10", "11", "100", "101"]`

**Big O.** O(N log N) time and space: N strings of about log N digits each, every one produced by appending a digit to an earlier one.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-4-20) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-4-20)

In [ ]:
# 2.4.20 Binary numbers 1 to n with a queue
def binary_numbers(n):
    raise NotImplementedError

In [ ]:
def test_ex_2_4_20():
    assert binary_numbers(5) == ['1', '10', '11', '100', '101']
    assert binary_numbers(0) == []
    assert binary_numbers(1) == ['1']
    assert binary_numbers(16)[-1] == '10000'
    assert binary_numbers(16) == [format(i, 'b') for i in range(1, 17)]


test_ex_2_4_20()
print('2.4.20 passed')